# Feature Engineering\n
\n
This notebook focuses on creating new features from the raw data to improve the performance of the risk prediction model.\n
\n
We will combine data from multiple sources, create aggregated features, and encode categorical variables.

In [ ]:
import pandas as pd\n
import numpy as np\n
from datetime import datetime\n
\n
# Load datasets\n
students_df = pd.read_csv('../datasets/students_sample.csv')\n
marks_df = pd.read_csv('../datasets/marks_sample.csv')\n
attendance_df = pd.read_csv('../datasets/attendance_sample.csv')\n
fees_df = pd.read_csv('../datasets/fees_sample.csv')\n
\n
print('Datasets loaded.')

## 1. Merge Datasets\n
\n
We'll merge the datasets on student_id to create a comprehensive feature set for each student.

In [ ]:
# Start with students_df as base\n
features_df = students_df.copy()\n
\n
# Merge with marks data (average marks per student)\n
marks_agg = marks_df.groupby('student_id').agg({\n
    'marks_obtained': 'mean',\n
    'marks_obtained': 'std',\n
    'exam_type': 'count'\n
}).rename(columns={\n
    'marks_obtained': 'avg_marks',\n
    'marks_obtained': 'std_marks',\n
    'exam_type': 'num_exams'\n
}).reset_index()\n
\n
features_df = features_df.merge(marks_agg, on='student_id', how='left')\n
\n
# Merge with attendance data (average attendance per student)\n
attendance_agg = attendance_df.groupby('student_id').agg({\n
    'percentage': 'mean',\n
    'percentage': 'std',\n
    'status': lambda x: (x == 'Present').mean()  # proportion of present days\n
}).rename(columns={\n
    'percentage': 'avg_attendance',\n
    'percentage': 'std_attendance',\n
    'status': 'prop_present'\n
}).reset_index()\n
\n
features_df = features_df.merge(attendance_agg, on='student_id', how='left')\n
\n
# Merge with fees data (fee status, balance, etc.)\n
fees_agg = fees_df.groupby('student_id').agg({\n
    'status': lambda x: (x == 'Paid').mean(),  # proportion of paid fees\n
    'balance': 'sum',\n
    'late_fine': 'sum'\n
}).rename(columns={\n
    'status': 'prop_paid_fees',\n
    'balance': 'total_balance',\n
    'late_fine': 'total_late_fine'\n
}).reset_index()\n
\n
features_df = features_df.merge(fees_agg, on='student_id', how='left')

## 2. Create New Features\n
\n
We'll create additional features that might be predictive of dropout risk.

In [ ]:
# Feature: age_group\n
features_df['age_group'] = pd.cut(features_df['age'], bins=[0, 18, 20, 22, 100], labels=['<18', '18-20', '20-22', '22+'])\n
\n
# Feature: course_risk (based on historical dropout rates per course - placeholder)\n
# In a real scenario, we would compute this from historical data\n
course_risk_map = {\n
    'B.Tech Computer Science': 0.1,\n
    'B.Tech Electronics': 0.2,\n
    'B.Tech Mechanical': 0.15,\n
    'B.Tech Civil': 0.25\n
}\n
features_df['course_risk'] = features_df['course'].map(course_risk_map).fillna(0.15)\n
\n
# Feature: fee_pressure (high balance relative to average)\n
features_df['fee_pressure'] = np.where(features_df['total_balance'] > features_df['total_balance'].median(), 1, 0)\n
\n
# Feature: academic_performance (combined marks and attendance)\n
features_df['academic_performance'] = (\n
    0.6 * features_df['avg_marks'] / 100 + \n
    0.4 * features_df['avg_attendance'] / 100\n
)\n
\n
# Feature: engagement (library books issued and complaints count - inverse)\n
features_df['engagement_score'] = (\n
    features_df['library_books_issued'] * 0.1 - \n
    features_df['complaints_count'] * 0.2\n
)\n
print('New features created.')

## 3. Handle Missing Values\n
\n
Check for missing values and decide on imputation strategy.

In [ ]:
print('Missing values before imputation:')\n
print(features_df.isnull().sum())\n
\n
# For numerical features, fill with median\n
num_cols = features_df.select_dtypes(include=[np.number]).columns\n
features_df[num_cols] = features_df[num_cols].fillna(features_df[num_cols].median())\n
\n
# For categorical features, fill with mode\n
cat_cols = features_df.select_dtypes(include=['object']).columns\n
for col in cat_cols:\n
    features_df[col] = features_df[col].fillna(features_df[col].mode()[0] if not features_df[col].mode().empty else 'Unknown')\n
\n
print('\nMissing values after imputation:')\n
print(features_df.isnull().sum())

## 4. Encode Categorical Variables\n
\n
Convert categorical variables into numerical format for modeling.

In [ ]:
from sklearn.preprocessing import LabelEncoder\n
\n
# Identify categorical columns\n
categorical_cols = features_df.select_dtypes(include=['object']).columns.tolist()\n
print('Categorical columns:', categorical_cols)\n
\n
# Apply Label Encoding\n
label_encoders = {}\n
for col in categorical_cols:\n
    le = LabelEncoder()\n
    features_df[col] = le.fit_transform(features_df[col])\n
    label_encoders[col] = le\n
\n
print('\nFeature engineering complete. Final feature set shape:', features_df.shape)\n
print('\nFirst few rows:')\n
display(features_df.head())

## 5. Save Engineered Features\n
\n
Save the engineered features for use in model training.

In [ ]:
# Select features for modeling (excluding student_id and target variable if present)\n
# We'll keep student_id for reference\n
model_features = features_df.copy()\n
\n
# Save to CSV\n
model_features.to_csv('../datasets/engineered_features.csv', index=False)\n
print('Engineered features saved to ../datasets/engineered_features.csv')